# <font color = 'red'> DEPENDENCIAS

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.miscmodels.ordinal_model import OrderedModel
from sklearn.preprocessing import MinMaxScaler

import sys
import os

# Agregar la carpeta calibration_code al path
sys.path.append(os.path.abspath("../../calibration_code"))

# Ahora puedes importar los módulos personalizados
from modelling_tools import (plot_histogram, plot_univariate_freq, assign_deciles, count_categories_by_decile, 
                             calculate_category_proportions, summarize_decile_analysis, summarize_grouped_deciles, group_deciles,
                             compute_odds_ratio)
from visualization_tools import plot_interactive_chart
from utils import g
from config import get_data_path, get_code_path
from data_cleaning import check_dataframe_quality

# <font color = 'red'> CARGA DE DATOS

In [2]:
df = pd.read_csv(get_data_path("bivariate_preprocessed_data.csv"))

In [3]:
res = check_dataframe_quality(df)

No missing values found.
No infinite values found.
No duplicate rows found.


# <font color = 'red'> ANÁLISIS

In [4]:
col = "Num_Bank_Accounts"

## <font color = 'skyblue'> ANÁLISIS GENERAL

La mediana del número de cuentas es significativamente mayor en el caso de los malos. Le sigue Standard y luego Good, lo cual es lo esperado.

In [5]:
fig_box = px.box(df, x="Credit_Mix", y=col, title=f"Distribution of {col} by Credit Score Category")
fig_box.show()

## <font color = 'skyblue'> ANÁLISIS POR DECILES

In [6]:
continuous_variable= col
decile_col_name = continuous_variable + '_Decile'
target_col_string = "Credit_Mix" # variable dependiente con nombres string
target_col = 'Credit_Score' # variable dependiente int (para modelos)

In [7]:
analysis_summary = summarize_decile_analysis(df, continuous_variable, decile_col_name, target_col_string)

# Obtener los resultados
df_deciles = analysis_summary["df_deciles"]  # DataFrame con los deciles asignados
deciles_summary = analysis_summary["decile_summary"]  # Resumen de deciles con conteos y proporciones
display(deciles_summary)
res = check_dataframe_quality(df_deciles)

,Decile_Min,Decile_Max,Decile_Count,Decile_Proportion,count_Bad,count_Good,count_Standard,prop_Bad,prop_Good,prop_Standard
Num_Bank_Accounts_Decile,,,,,,,,,,
0,0,2,13298,0.13298,0,13247,51,0.000000,0.996165,0.003835
1,3,3,12106,0.12106,0,5190,6916,0.000000,0.428713,0.571287
2,4,4,12342,0.12342,0,5059,7283,0.000000,0.409901,0.590099
3,5,5,12299,0.12299,13,4953,7333,0.001057,0.402716,0.596227
4,6,6,13177,0.13177,4786,671,7720,0.363209,0.050922,0.585869
5,7,7,12996,0.12996,4971,498,7527,0.382502,0.038319,0.579178
6,8,8,12943,0.12943,4757,759,7427,0.367535,0.058642,0.573824
7,9,9,5501,0.05501,4689,7,805,0.852390,0.001272,0.146337
8,10,11,5338,0.05338,4552,0,786,0.852754,0.000000,0.147246


No missing values found.
No infinite values found.
No duplicate rows found.


In [33]:
df[(df[continuous_variable] >= 5309.226667) & (df[continuous_variable] <= 6712.043333) & (df['Credit_Score'] == 2)].shape

(2728, 85)

Porporción de Buenos: aunque se observa una relación positiva entre los ingresos netos y la proporción de buenos, se observa un
cambio abrupto entre los deciles 6 y 7, al pasar de una proporción de buenos de 41% a 15%,
lo cual no es razonable.

Proporción de standard: aunque sí existe una tendencia negativa entre la proporción de Standard y los ingresos
hay cambios irregulares que son poco razonables.

Proporción de malos: de manera similar, la proporción de malos y los ingresos tienen una relación inversa, sin embargo, 
hay cambios irregulares abruptos.

In [8]:
chart_types = {
    "prop_Good":"line",
    "prop_Standard":"line",
    "prop_Bad": "line",  
    "Decile_Count": "bar"    
}

fig = plot_interactive_chart(
    df=deciles_summary,  
    y_columns=["prop_Bad", "prop_Good", "prop_Standard", "Decile_Count"],  
    x_column="Decile_Max",  
    chart_types=chart_types, 
    title=f"Proportion of Credit Score Categories by {continuous_variable}",
    x_title="Decile",
    y_title="Decile Count",  
    y2_title="Proportion",   
    secondary_y=["prop_Bad", "prop_Good", "prop_Standard"],  
    width=900,
    height=500,
    custom_colors={"prop_Bad": "red", "prop_Good":"lightgreen",
    "prop_Standard":"brown", "Decile_Count": "gray"}  
)

fig.show()

<font color = 'brown'> Agrupación de deciles

Se agrupan deciles buscando una relación monótona entre las proporciones y los ingresos:

In [10]:
group_map = {0: "Group_1", 
             1: "Group_1", 
             2: "Group_1", 
             3: "Group_1", 
             4: "Group_2",
             5: "Group_2", 
             6: "Group_2", 
             7: "Group_3", 
             8: "Group_3"}

# Agrupar los deciles
grouped_col_name = "Grouped_" + continuous_variable

df_deciles_grouped = group_deciles(df_deciles, decile_col_name, grouped_col_name, group_map)

summary_results = summarize_grouped_deciles(df_deciles_grouped, grouped_col_name, continuous_variable, target_col_string, prefix="Decile_")
grouped_deciles_summary = summary_results['df']


# Definir el mapeo manual de los grupos a enteros
group_mapping = {
    'Group_1': 1,
    'Group_2': 2,
    'Group_3': 3
}

if not set(group_map.values()) == set(group_mapping.keys()):
    print("Problemas en el mapeo de grupos a enteros!")

# Usar `.map()` en lugar de `.replace()` para evitar el warning
df_deciles_grouped[grouped_col_name] = (
    df_deciles_grouped[grouped_col_name]
    .map(group_mapping)  # Mapear los valores
    .astype("Int64")      # Convertir a entero manejando NaN si existen
)

display(grouped_deciles_summary)

res = check_dataframe_quality(df_deciles_grouped)

,Decile_Min,Decile_Max,Decile_Count,Decile_Proportion,count_Bad,count_Good,count_Standard,prop_Bad,prop_Good,prop_Standard
Grouped_Num_Bank_Accounts,,,,,,,,,,
Group_1,0,5,50045,0.50045,13,28449,21583,0.000260,0.568468,0.431272
Group_2,6,8,39116,0.39116,14514,1928,22674,0.371050,0.049289,0.579660
Group_3,9,11,10839,0.10839,9241,7,1591,0.852569,0.000646,0.146785


No missing values found.
No infinite values found.
No duplicate rows found.


In [11]:
chart_types = {
    "prop_Good":"line",
    "prop_Standard":"line",
    "prop_Bad": "line",  
    "Decile_Count": "bar"    
}

fig = plot_interactive_chart(
    df=grouped_deciles_summary,  
    y_columns=["prop_Bad", "prop_Good", "prop_Standard", "Decile_Count"],  
    x_column="Decile_Max",  
    chart_types=chart_types, 
    title=f"Proportion of Credit Score Categories by {continuous_variable}",
    x_title="Decile",
    y_title="Decile Count",  
    y2_title="Proportion",   
    secondary_y=["prop_Bad", "prop_Good", "prop_Standard"],  
    width=900,
    height=500,
    custom_colors={"prop_Bad": "red", "prop_Good":"lightgreen",
    "prop_Standard":"brown", "Decile_Count": "gray"}  
)

fig.show()

## <font color = 'skyblue'> REGRESIONES BIVARIADAS

Como Credit_Score tiene tres categorías (Bad, Standard, Good) se pueden usar dos enfoques 
de regresión categórica: 

- Regresión Logística Multinomial → No asume orden en las categorías (como si fueran colores: rojo, azul, verde).
- Regresión Logística Ordinal → Asume que hay un orden en las categorías (Bad < Standard < Good).

Dado que hay un orden entre las categorías se utiliza Regresión Logística Ordinal:

<font color = 'gold'> Sin Agrupaciones

In [12]:
df_ = df_deciles_grouped.copy()
x_variable = continuous_variable
res = check_dataframe_quality(df_)

No missing values found.
No infinite values found.
No duplicate rows found.


In [13]:
# para no generar problemas numéricos debe escalarse esta variable.
# se opta por normalizar la variable:

g(df_[[x_variable]].describe()).transpose()

,count,mean,std,min,25%,50%,75%,max
Num_Bank_Accounts,"100,000.00",5.37,2.59,0.00,3.00,5.00,7.00,11.00


Todos los coeficientes son significativos.

Num_Bank_Accounts_Scaled    -9.4764: según lo esperado, el coeficiente es negativo: por cada número de cuenta adicional, la probabilidad de estar en una categoría superior de Credit_Score disminuye.

Threshold 0/1 -6.7353: Umbral que separa las categorías Bad y Standard. Si la puntuación supera este umbral, es más probable que 
sea standard en lugar de bad.

Threshold 1/2 1.2841: Umbral que separa las categorías Standard y Good. Si la puntuación supera este umbral, es más probable que 
sea Good en lugar de Standard.

In [17]:
df_ = df_deciles_grouped.copy()
x_variable = continuous_variable

scaler = MinMaxScaler()
df_[continuous_variable + "_Scaled"] = scaler.fit_transform(df_[[x_variable]])

res = check_dataframe_quality(df_)

# Ajustar el modelo de regresión logística ordinal con la variable escalada
model_income = OrderedModel(df_[target_col], df_[continuous_variable + "_Scaled"], distr="logit")
result_income = model_income.fit(method='bfgs')

# Mostrar resumen del modelo
print(result_income.summary())

# Calcular e interpretar el Odds Ratio
res_odds = compute_odds_ratio(result_income, variable_name=continuous_variable + "_Scaled", description="Ingreso Anual Escalado")
print(res_odds["interpretation"])


No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 0.699354
         Iterations: 13
         Function evaluations: 15
         Gradient evaluations: 15
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:                -69935.
Model:                   OrderedModel   AIC:                         1.399e+05
Method:            Maximum Likelihood   BIC:                         1.399e+05
Date:                Sat, 29 Mar 2025                                         
Time:                        17:48:58                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                               coef    st

<font color = 'gold'> Por Deciles

Todos los coeficientes son significativos.

Num_Bank_Accounts_Decile -0.8944: Por cada decil adicional, la probabilidad de estar en una categoría superior de Credit_Score disminuye.

Threshold 0/1 -0.2910: Umbral que separa las categorías Bad y Standard. Si la puntuación supera este umbral, es más probable que 
sea standard en lugar de bad.

Threshold 1/2 0.7741: Umbral que separa las categorías Standard y Good. Si la puntuación supera este umbral, es más probable que 
sea Good en lugar de Standard.

In [19]:
df_ = df_deciles_grouped.copy()
x_variable = decile_col_name

res = check_dataframe_quality(df_)

model_age_decile = OrderedModel(df_[target_col], df_[x_variable], distr="logit")

result_age_decile = model_age_decile.fit(method='bfgs')

print(result_age_decile.summary())

res_odds = compute_odds_ratio(result_age_decile, variable_name = x_variable, description = 'Decil de Ingreso Neto')
print(res_odds['interpretation'])

No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 0.711840
         Iterations: 12
         Function evaluations: 14
         Gradient evaluations: 14
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:                -71184.
Model:                   OrderedModel   AIC:                         1.424e+05
Method:            Maximum Likelihood   BIC:                         1.424e+05
Date:                Sat, 29 Mar 2025                                         
Time:                        18:05:29                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                               coef    st

<font color = 'gold'> Por Agrupamientos de Deciles

Todos los coeficientes son significativos.

Grouped_Num_Bank_Accounts -3.2364: Por cada grupo adcional, la probabilidad de estar en una categoría superior de Credit_Score disminuye.

Threshold 0/1 0.7172: Umbral que separa las categorías Bad y Standard.

Threshold 1/2	0.7862: Umbral que separa las categorías Standard y Good. Si la puntuación supera 0.7604, es más probable que 
sea Good en lugar de Standard.

In [22]:
df_ = df_deciles_grouped.copy()
x_variable = grouped_col_name

df_[target_col] = df_[target_col].astype(int)
df_[x_variable] = df_[x_variable].astype(int)

res = check_dataframe_quality(df_)

model_age_decile = OrderedModel(df_[target_col], df_[x_variable], distr="logit")

result_age_decile = model_age_decile.fit(method='bfgs')

print(result_age_decile.summary())

res_odds = compute_odds_ratio(result_age_decile, variable_name = x_variable, description = 'Decil de Edad')
print(res_odds['interpretation'])

No missing values found.
No infinite values found.
No duplicate rows found.
Optimization terminated successfully.
         Current function value: 0.728712
         Iterations: 14
         Function evaluations: 15
         Gradient evaluations: 15
                             OrderedModel Results                             
Dep. Variable:           Credit_Score   Log-Likelihood:                -72871.
Model:                   OrderedModel   AIC:                         1.457e+05
Method:            Maximum Likelihood   BIC:                         1.458e+05
Date:                Sat, 29 Mar 2025                                         
Time:                        18:09:12                                         
No. Observations:              100000                                         
Df Residuals:                   99997                                         
Df Model:                           1                                         
                                coef    s

## <font color = 'skyblue'> CONCLUSIONES

### 📊 Comparación de Representaciones de `Num_Bank_Accounts` usando Regresión Ordinal

| Representación                     | Coeficiente principal | Indicadores de ajuste                                                                 | Interpretación                                                                 |
|------------------------------------|------------------------|----------------------------------------------------------------------------------------|--------------------------------------------------------------------------------|
| `Num_Bank_Accounts_Scaled`        | -9.4764               | **Log-Likelihood**: -69,935<br>**AIC**: 139,871<br>**BIC**: 139,913                   | 🔹 Mejor ajuste global.<br>🔹 El coeficiente negativo sugiere que más cuentas bancarias se asocian con menor `Credit_Score`. Escalado previo asegura estabilidad numérica. |
| `Num_Bank_Accounts_Decile`        | -0.8944               | **Log-Likelihood**: -71,184<br>**AIC**: 142,371<br>**BIC**: 142,412                   | 🔹 Ajuste ligeramente inferior.<br>🔹 Discretizar en deciles reduce información pero mejora interpretabilidad. |
| `Grouped_Num_Bank_Accounts`       | -3.2364               | **Log-Likelihood**: -72,871<br>**AIC**: 145,742<br>**BIC**: 145,783                   | 🔹 Peor ajuste global.<br>🔹 Aunque más interpretable, agrupar reduce considerablemente la capacidad explicativa. |



## <font color = 'skyblue'> EXPORTACIÓN DE DATOS CON VARIABLES ADICIONALES

In [42]:
# df_deciles_grouped.to_csv("../../calibration_data/preprocessed_data.csv", index=False)
